In [1]:
# Group B - Assignment 3
# Problem Statement:
# Given a bank customer, build a neural network-based classifier that can determine whether they will leave or not in the next 6 months.
# Dataset Description: The case study is from an open-source dataset from Kaggle.
# The dataset contains 10,000 sample points with 14 distinct features such as CustomerId, CreditScore, Geography, Gender, Age, Tenure, Balance, etc.
# Perform following steps:
# 1. Read the dataset.
# 2. Distinguish the feature and target set and divide the data set into training and test sets.
# 3. Normalize the train and test data.
# 4. Initialize and build the model. Identify the points of improvement and implement the same.
# 5. Print the accuracy score and confusion matrix (5 points).

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("All required libraries imported successfully.")

All required libraries imported successfully.
TensorFlow version: 2.21.0


In [2]:
print("=== Step 1: Read the Dataset ===")
df = pd.read_csv("Churn_Modelling.csv")

print("\n--- Initial Dataset Head ---")
print(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Dataset Shape ---")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

=== Step 1: Read the Dataset ===

--- Initial Dataset Head ---
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2       

In [3]:
print("=== Data Pre-processing ===")

# Drop irrelevant metadata columns
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

# Encode Categorical Variables (Gender & Geography)
le_gender = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender'])

# One-hot encoding for Geography
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

print("\n--- Preprocessed Dataset Head ---")
print(df.head())

=== Data Pre-processing ===

--- Preprocessed Dataset Head ---
   CreditScore  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619       0   42       2       0.00              1          1   
1          608       0   41       1   83807.86              1          0   
2          502       0   42       8  159660.80              3          1   
3          699       0   39       1       0.00              2          0   
4          850       0   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_Germany  Geography_Spain  
0               1        101348.88       1              False            False  
1               1        112542.58       0              False             True  
2               0        113931.57       1              False            False  
3               0         93826.63       0              False            False  
4               1         79084.10       0              False             T

In [4]:
print("=== Step 2 & 3: Feature-Target Division & Normalization ===")

# Distinguish feature set (X) and target set (y)
X = df.drop('Exited', axis=1)
y = df['Exited']

# Split dataset into training and testing sets (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Normalize the train and test data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape: ", X_test_scaled.shape)

=== Step 2 & 3: Feature-Target Division & Normalization ===
X_train shape: (8000, 11)
X_test shape:  (2000, 11)


In [5]:
print("=== Step 4: Build and Train Neural Network Model ===")

# Initialize Sequential ANN Model
model = Sequential([
    Dense(units=16, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dropout(0.2),  # Dropout layer to prevent overfitting
    Dense(units=8, activation='relu'),
    Dropout(0.2),
    Dense(units=1, activation='sigmoid')  # Binary output
])

# Compile the Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n--- Neural Network Architecture ---")
model.summary()

# Train the Model
print("\n--- Model Training ---")
history = model.fit(
    X_train_scaled, 
    y_train, 
    batch_size=32, 
    epochs=25, 
    validation_split=0.2, 
    verbose=1
)

=== Step 4: Build and Train Neural Network Model ===


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Neural Network Architecture ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 16)                  │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 337 (1.32 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 0 (0.00 B)


--- Model Training ---
Epoch 1/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6742 - loss: 0.6418 - val_accuracy: 0.7944 - val_loss: 0.5027
Epoch 2/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7920 - loss: 0.5020 - val_accuracy: 0.8019 - val_loss: 0.4499
Epoch 3/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7986 - loss: 0.4726 - val_accuracy: 0.8081 - val_loss: 0.4312
Epoch 4/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8039 - loss: 0.4587 - val_accuracy: 0.8119 - val_loss: 0.4213
Epoch 5/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8056 - loss: 0.4504 - val_accuracy: 0.8181 - val_loss: 0.4140
Epoch 6/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8130 - loss: 0.4400 - val_accuracy: 0.8263 - val_loss: 0.4055
Epoch 7/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8119 - loss: 0.4348 - val_accuracy: 0.8319 - val_loss: 0.3989
Epoch 8/25
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8245 - loss: 0

In [6]:
print("=== Step 5: Evaluation, Accuracy Score, and Confusion Matrix ===")

# Make predictions on test set
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute Metrics
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n--- Model Evaluation Results ---")
print("Accuracy Score:", acc)

print("\n--- Confusion Matrix ---")
print(cm)

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

=== Step 5: Evaluation, Accuracy Score, and Confusion Matrix ===
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

--- Model Evaluation Results ---
Accuracy Score: 0.8585

--- Confusion Matrix ---
[[1576   31]
 [ 252  141]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.86      0.98      0.92      1607
           1       0.82      0.36      0.50       393

    accuracy                           0.86      2000
   macro avg       0.84      0.67      0.71      2000
weighted avg       0.85      0.86      0.84      2000

